# 04. Census, housing and deprivation attributes

This notebook attaches population, housing and deprivation attributes to the
census blocks and tracts built by the preceding notebook.

Population is the weight applied to every travel time computed later, so the
block level counts determine what the access measures describe. Vehicle
availability decides whether a distance that is long on foot is a binding
constraint on a household. Deprivation is the social dimension the access
measures are compared against.

Three properties of the census distribution govern how it is read.

Results are published as one archive per federal entity, and each archive holds
rows at several levels of aggregation in a single file. A state total, a
municipality total, a locality total and a tract total sit alongside the
individual blocks, distinguished by zeros in the identifier fields. The block
rows are separated from the totals, and the tract totals are retained for
comparison against the sum of their blocks.

Counts below three are suppressed for confidentiality and published as an
asterisk. The true value is zero, one or two, so this is interval censoring
rather than missing data. Suppressed cells take the midpoint of that interval
and the number affected is reported for every variable.

Archives differ in character encoding, and a member of a compressed archive
cannot be read a second time from its beginning once reading has started. Each
member is therefore held in memory before being decoded, so that decoding can
be attempted in more than one encoding.

The urban restriction follows from the data rather than from a filter applied
here. The census of population, the deprivation measures and the tract
definitions all cover urban areas, so a block that does not appear in the
census lies outside that universe.

Outputs are `Results/04_blocks.parquet`, one row per block inside the urban
universe, and `Results/04_tracts.parquet`, one row per tract with population,
housing, vehicle availability and both deprivation measures.

Runtime is roughly half an hour, most of it reading the archives. Each entity
is cached as it completes, so an interrupted run resumes where it stopped.

## 1. Setup

In [ ]:
import sys, os, glob, io, re, zipfile, hashlib, importlib

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT = os.environ.get('MXO_BASE', '/content/drive/MyDrive/Mexico_Obesity')
os.environ['MXO_BASE'] = PROJECT
sys.path.insert(0, f'{PROJECT}/Code')
import mxo_utils as mx
importlib.reload(mx)

assert mx.BASE == PROJECT, f'the module resolves to {mx.BASE}'

import numpy as np
import pandas as pd

rep = mx.Report('04_census_deprivation').start()
mx.flow_reset()

INPUTS = mx.DIRS['inputs']

BLOCKS = pd.read_parquet(os.path.join(mx.DIRS['results'], '03_blocks.parquet'))
print(f'{len(BLOCKS):,} blocks in {BLOCKS.CVEGEO.nunique():,} tracts '
      f'across {BLOCKS.slug.nunique()} metropolitan areas')

WANTED = set(BLOCKS['CVEGEO_MZA'])
ENTITIES = sorted(BLOCKS['CVE_ENT'].unique())
print(f'{len(ENTITIES)} federal entities contain at least one block')

## 2. Reading one archive

The variables read are the population total, the age and sex detail, the
several published counts of dwellings, the count of dwellings reporting a car,
and mean years of schooling. The archives carry 230 variables, so reading a
named subset keeps memory use modest.

The data member is selected by name. Archives also contain a data dictionary, a
metadata note and in some entities a log of revisions, and the revision log is
a comma separated file of comparable size, so selecting the largest member
would not reliably select the data.

In [ ]:
NEEDED = ['ENTIDAD', 'MUN', 'LOC', 'AGEB', 'MZA', 'POBTOT']

OPTIONAL = ['POBFEM', 'POBMAS', 'POB0_14', 'POB65_MAS',
            'VIVTOT', 'TVIVHAB', 'TVIVPAR', 'VIVPAR_HAB', 'VIVPARH_CV',
            'TVIVPARHAB', 'VPH_AUTOM', 'GRAPROES']

# The cache directory is keyed by the set of variables read, so a change to
# that set yields a different directory and no file built from another
# selection can be read back into this run.
FIELD_KEY = hashlib.blake2b(','.join(NEEDED + OPTIONAL).encode(),
                            digest_size=4).hexdigest()
CACHE = mx.cache_dir(f'cache_04_census_{FIELD_KEY}')
print(f'variables read: {len(NEEDED + OPTIONAL)}')
print('cache directory:', CACHE)

DATA_MEMBER = re.compile(r'conjunto_de_datos_ageb_urbana_\d{2}_cpv2020\.csv$', re.I)
OTHER_MEMBER = re.compile(r'bitacora|diccionario|metadatos', re.I)


def read_entity(path):
    """Census rows for one federal entity, restricted to the named variables."""
    with zipfile.ZipFile(path) as zf:
        names = [n for n in zf.namelist() if not n.endswith('/')]
        members = [n for n in names
                   if DATA_MEMBER.search(n) and not OTHER_MEMBER.search(n)]
        if len(members) != 1:
            raise RuntimeError(
                f'{os.path.basename(path)}: expected one data member, found '
                f'{len(members)}: {members}')
        raw = zf.read(members[0])

    head = None
    for enc in ('utf-8', 'latin1'):
        try:
            head = pd.read_csv(io.BytesIO(raw), nrows=0, encoding=enc)
            encoding = enc
            break
        except UnicodeDecodeError:
            continue
    if head is None:
        raise UnicodeDecodeError('utf-8', b'', 0, 1,
                                 f'{os.path.basename(path)} will not decode')

    missing = [col for col in NEEDED if col not in head.columns]
    if missing:
        raise KeyError(f'{os.path.basename(path)} lacks {missing} among its '
                       f'{len(head.columns)} variables')

    use = NEEDED + [col for col in OPTIONAL if col in head.columns]
    df = pd.read_csv(io.BytesIO(raw), usecols=use, dtype=str, encoding=encoding)
    df['_encoding'] = encoding
    df['_n_fields'] = len(head.columns)
    return df

## 3. Separating blocks from aggregate rows

A row describes a block when its block number is other than zero. A row with a
zero block number is a total, for the tract when the tract and locality fields
are populated, and for successively larger areas as those fields fall to zero
in turn.

In [ ]:
def split_levels(df):
    """Block rows and tract total rows from one entity's file."""
    mza = df['MZA'].astype(str).str.strip().str.zfill(3)
    ageb = df['AGEB'].astype(str).str.strip().str.zfill(4)
    loc = df['LOC'].astype(str).str.strip().str.zfill(4)

    is_block = mza.ne('000')
    is_tract_total = mza.eq('000') & ageb.ne('0000') & loc.ne('0000')

    return (df[is_block].copy(), df[is_tract_total].copy(),
            int(is_block.sum()), int(is_tract_total.sum()))

In [ ]:
census_parts, total_parts, level_rows = [], [], []

zips = sorted(glob.glob(os.path.join(INPUTS, 'Census', '*.zip')))
assert len(zips) == 32, f'{len(zips)} archives found, expected 32'

for i, z in enumerate(zips, 1):
    ent = re.search(r'urbana_(\d{2})_', os.path.basename(z)).group(1)
    cpath = os.path.join(CACHE, f'ent{ent}.parquet')
    tpath = os.path.join(CACHE, f'ent{ent}_tracts.parquet')

    if os.path.exists(cpath) and os.path.exists(tpath):
        census_parts.append(pd.read_parquet(cpath))
        total_parts.append(pd.read_parquet(tpath))
        continue

    with mx.stage(f'[{i:>2}/32] entity {ent}'):
        df = read_entity(z)
        blocks, totals, n_b, n_t = split_levels(df)

        blocks['CVEGEO_MZA'] = mx.build_cvegeo_mza(
            blocks['ENTIDAD'], blocks['MUN'], blocks['LOC'],
            blocks['AGEB'], blocks['MZA']).values
        keep = blocks[blocks['CVEGEO_MZA'].isin(WANTED)].copy()

        totals['CVEGEO'] = mx.build_cvegeo(
            totals['ENTIDAD'], totals['MUN'], totals['LOC'],
            totals['AGEB']).values
        totals = totals[['CVEGEO', 'POBTOT']].copy()

        level_rows.append({'entity': ent, 'variables': int(df['_n_fields'].iloc[0]),
                           'encoding': df['_encoding'].iloc[0],
                           'rows': len(df), 'block_rows': n_b,
                           'tract_totals': n_t, 'in_frame': len(keep)})
        keep.to_parquet(cpath, index=False)
        totals.to_parquet(tpath, index=False)

    census_parts.append(keep)
    total_parts.append(totals)

CEN = pd.concat(census_parts, ignore_index=True)
TOT = pd.concat(total_parts, ignore_index=True)
LEVELS = pd.DataFrame(level_rows)
print(f'\n{len(CEN):,} census block rows inside the metropolitan frame')
if len(LEVELS):
    print()
    print(LEVELS.to_string(index=False))
    rep.add_table('archives as read', LEVELS, max_rows=32)

## 4. Converting the published counts

Each variable is parsed so that a suppressed cell and a cell for an area that
was not enumerated remain distinguishable. Suppressed cells take the value one,
the midpoint of the interval they stand for. The share of cells affected is
reported for each variable, because a variable suppressed in a large fraction
of blocks will not support a precise statement.

In [ ]:
NUMERIC = [c for c in ['POBTOT'] + OPTIONAL if c in CEN.columns]
CEN, parse_report = mx.parse_census_frame(CEN, NUMERIC, star_fill=1.0)

print(parse_report.to_string(index=False))
rep.add_table('suppression by variable', parse_report, max_rows=20)

star_cols = [c for c in CEN.columns if c.startswith('_star_')]
nd_cols = [c for c in CEN.columns if c.startswith('_nd_')]
print(f'\nsuppressed cells, all variables     {int(CEN[star_cols].sum().sum()):,}')
print(f'cells in unenumerated areas          {int(CEN[nd_cols].sum().sum()):,}')

## 5. Joining blocks to the frame

In [ ]:
cols = ['CVEGEO_MZA'] + NUMERIC
B = BLOCKS.merge(CEN[cols].drop_duplicates('CVEGEO_MZA'),
                 on='CVEGEO_MZA', how='left', indicator='_src')

matched = B['_src'].eq('both')
print(f'blocks in the geographic frame        {len(B):,}')
print(f'blocks present in the urban census    {int(matched.sum()):,} '
      f'({matched.mean():.1%})')
print(f'blocks outside the urban universe     {int((~matched).sum()):,}')

print('\nareas contributing the most blocks outside the urban universe:')
print(B.loc[~matched].groupby('slug').size()
       .sort_values(ascending=False).head(12).to_string())

B = B[matched].drop(columns='_src').copy()
B['POBTOT'] = B['POBTOT'].fillna(0.0)

## 6. Block sums against the published tract totals

The census publishes a total for each tract as well as the counts for its
blocks, so the sum of a tract's blocks can be compared with the total published
for that tract. Two things separate the two quantities. Suppression replaces
counts below three with an interval midpoint, which moves a sum in either
direction by a small amount. A tract crossed by the boundary of a metropolitan
area keeps only those of its blocks whose centres fall inside, which lowers the
sum and can lower it a great deal.

The share of each tract's published population that lies inside the frame is
retained as a variable, so that tract level results can later be examined on
the subset that is substantially complete.

In [ ]:
blocksum = B.groupby('CVEGEO')['POBTOT'].sum().rename('block_sum')
TOTn, _, _ = mx.parse_census_column(TOT['POBTOT'], star_fill=1.0)
published = (TOT.assign(pub=TOTn.values).dropna(subset=['pub'])
               .drop_duplicates('CVEGEO').set_index('CVEGEO')['pub'])

cmp = pd.concat([blocksum, published.rename('published')], axis=1, join='inner')
cmp['diff'] = cmp['block_sum'] - cmp['published']
cmp['completeness'] = (cmp['block_sum'] /
                       cmp['published'].replace(0, np.nan)).round(4)

print(f'tracts with a published total to compare  {len(cmp):,}')
print(f'sums that agree exactly                   {int((cmp["diff"] == 0).sum()):,} '
      f'({(cmp["diff"] == 0).mean():.1%})')
print(f'sums above the published total            {int((cmp["diff"] > 0).sum()):,}')
print(f'sums below the published total            {int((cmp["diff"] < 0).sum()):,}')

print('\nshare of the published population inside the frame:')
print(cmp['completeness'].describe(percentiles=[0.01, 0.05, 0.25, 0.5]).to_string())
for cut in (0.99, 0.95, 0.90, 0.50):
    n = int((cmp['completeness'] < cut).sum())
    print(f'   tracts below {cut:.0%} complete: {n:,} ({n / len(cmp):.2%})')

worst = cmp.reindex(cmp['completeness'].sort_values().index).head(10)
print('\nthe most truncated tracts:')
print(worst.to_string())
rep.add_table('most truncated tracts', worst.reset_index(), max_rows=20)

## 7. Tracts with blocks in two metropolitan areas

Where two metropolitan areas adjoin, a tract can have blocks on both sides of
the boundary. Tract level reporting requires each tract to belong to one area,
so a tract in that position is assigned in full to the area holding the greater
part of its population.

In [ ]:
span = B.groupby('CVEGEO')['slug'].nunique()
split = span[span > 1].index
print(f'tracts with blocks in more than one area: {len(split)}')

if len(split):
    detail = (B[B.CVEGEO.isin(split)].groupby(['CVEGEO', 'slug'])
                .agg(blocks=('CVEGEO_MZA', 'size'), pop=('POBTOT', 'sum'))
                .reset_index())
    print(detail.to_string(index=False))

    winner = (detail.sort_values(['CVEGEO', 'pop', 'blocks'],
                                 ascending=[True, False, False])
                    .drop_duplicates('CVEGEO').set_index('CVEGEO')['slug'])
    moved = B['CVEGEO'].isin(split)
    B.loc[moved, 'slug'] = B.loc[moved, 'CVEGEO'].map(winner).values
    print(f'\n{int(moved.sum()):,} blocks reassigned so that each tract '
          'belongs to one area')
    rep.add_table('tracts spanning two areas', detail, max_rows=40)
    assert B.groupby('CVEGEO')['slug'].nunique().max() == 1

## 8. Aggregating to tracts

In [ ]:
agg = {c: 'sum' for c in NUMERIC if c != 'GRAPROES'}
agg['area_m2'] = 'sum'

T = (B.groupby('CVEGEO')
       .agg(blocks=('CVEGEO_MZA', 'size'),
            blocks_populated=('POBTOT', lambda s: int((s > 0).sum())),
            slug=('slug', 'first'),
            CVE_ENT=('CVE_ENT', 'first'),
            macro_region=('macro_region', 'first'),
            **{k: (k, v) for k, v in agg.items()})
       .reset_index())

T['pop_density_km2'] = (T['POBTOT'] / (T['area_m2'] / 1e6)).round(1)
T = T.merge(cmp[['completeness']].rename(columns={'completeness': 'tract_complete'}),
            left_on='CVEGEO', right_index=True, how='left')

print(f'{len(T):,} tracts, {T.POBTOT.sum():,.0f} people, {len(B):,} blocks')

## 9. Vehicle availability

The census publishes the number of dwellings reporting a car alongside several
counts of dwellings that differ in what they include. The housing
characteristics are tabulated over the dwellings for which those
characteristics were collected, so that count is the matching denominator for a
share of dwellings with a car.

The requirement is verifiable. A count of dwellings reporting a car cannot
exceed the number of dwellings it is drawn from, so a candidate denominator the
vehicle count exceeds is not the universe of the tabulation. Each published
count is reported with the share it produces and the number of tracts in which
it is exceeded.

In [ ]:
CANDIDATES = [c for c in ['VIVPARH_CV', 'TVIVPARHAB', 'TVIVHAB', 'VIVPAR_HAB',
                          'TVIVPAR', 'VIVTOT'] if c in T.columns]
assert 'VPH_AUTOM' in T.columns

rows = []
for d in CANDIDATES:
    r = T['VPH_AUTOM'] / T[d].replace(0, np.nan)
    rows.append({'denominator': d,
                 'tracts_exceeding_one': int((r > 1).sum()),
                 'largest_ratio': round(float(r.max()), 4),
                 'median_share_with_car': round(float(r.median()), 4),
                 'pooled_share_with_car': round(
                     float(T['VPH_AUTOM'].sum() / T[d].sum()), 4),
                 'dwellings_total': int(T[d].sum())})
DEN = pd.DataFrame(rows)
print(DEN.to_string(index=False))

DENOM = 'VIVPARH_CV'
assert DENOM in T.columns, DENOM
exceeded = int((T['VPH_AUTOM'] / T[DENOM].replace(0, np.nan) > 1).sum())
assert exceeded == 0, (
    f'{DENOM} is exceeded by the vehicle count in {exceeded} tracts and so is '
    'not the universe of the tabulation')

T['veh_share'] = (T['VPH_AUTOM'] / T[DENOM].replace(0, np.nan)).round(4)
T['veh_denominator'] = DENOM
print(f'\ndenominator in use: {DENOM}, dwellings for which housing '
      'characteristics were collected')
rep.add_table('candidate denominators', DEN, max_rows=10)

The share of households without a car enters the interpretation of walking
distance, so its sensitivity to specification is worth stating. Three choices
affect it: which published dwelling count serves as the denominator, which
tracts are included, and whether tracts are counted equally or weighted by
their population.

In [ ]:
def carless(frame, col='veh_share', weight=None):
    """Share of dwellings without a car at the median tract or median person."""
    f = frame.dropna(subset=[col])
    if weight is None:
        return 1 - float(f[col].median())
    order = f.sort_values(col)
    cw = order[weight].cumsum() / order[weight].sum()
    return 1 - float(order.loc[cw >= 0.5, col].iloc[0])


complete = T['tract_complete'].fillna(1.0) >= 0.99
specs = [('all tracts, counted equally', carless(T)),
         ('all tracts, weighted by population', carless(T, weight='POBTOT')),
         ('tracts at least 99 per cent complete, counted equally',
          carless(T[complete])),
         ('tracts at least 99 per cent complete, weighted by population',
          carless(T[complete], weight='POBTOT'))]
for d in CANDIDATES:
    T['_alt'] = T['VPH_AUTOM'] / T[d].replace(0, np.nan)
    specs.append((f'all tracts, counted equally, denominator {d}',
                  carless(T, col='_alt')))
T = T.drop(columns='_alt')

SENS = pd.DataFrame(specs, columns=['specification', 'share_without_a_car'])
SENS['share_without_a_car'] = SENS['share_without_a_car'].round(4)
print(SENS.to_string(index=False))
print(f'\nrange across specifications: '
      f'{SENS.share_without_a_car.min():.3f} to '
      f'{SENS.share_without_a_car.max():.3f}')
rep.add_table('share without a car under each specification', SENS, max_rows=20)

print('\nby completeness, region and deprivation grade:\n')
print(T.assign(group=np.where(complete, 'complete', 'truncated'))
       .groupby('group').agg(tracts=('CVEGEO', 'size'),
                             median_share_with_car=('veh_share', 'median'),
                             population=('POBTOT', 'sum')).round(4).to_string())
print()
print(T.groupby('macro_region')
       .agg(tracts=('CVEGEO', 'size'),
            median_share_with_car=('veh_share', 'median'),
            population=('POBTOT', 'sum')).round(4).to_string())

## 10. Tracts that cannot carry a population weighted measure

A tract whose blocks inside the metropolitan boundary are all uninhabited has
no origin from which to measure travel time. Quantities weighted by block
population are undefined for such a tract rather than zero, so it is marked
here and excluded by name wherever blocks are weighted.

In [ ]:
T['block_weightable'] = T['blocks_populated'] > 0
n_bad = int((~T['block_weightable']).sum())

print(f'tracts with no inhabited block inside the boundary: {n_bad}')
if n_bad:
    print(f'their population inside the frame: '
          f'{int(T.loc[~T.block_weightable, "POBTOT"].sum()):,}')
    print()
    print(T.loc[~T.block_weightable,
                ['CVEGEO', 'slug', 'blocks', 'POBTOT', 'tract_complete']]
           .head(15).to_string(index=False))
    rep.add_table('tracts without an inhabited block',
                  T.loc[~T.block_weightable,
                        ['CVEGEO', 'slug', 'blocks', 'POBTOT', 'tract_complete']],
                  max_rows=100)

print(f'\ntracts available for population weighted measures: '
      f'{int(T.block_weightable.sum()):,} of {len(T):,}')

## 11. Deprivation

Two published measures are joined at tract level. They differ in coverage and
in the direction of their scales, and both are oriented here so that a higher
value means more deprived.

In [ ]:
DEP_DIR = f'{INPUTS}/Deprivation'
imu_path = os.path.join(DEP_DIR, 'imu_2020.csv')
grs_path = os.path.join(DEP_DIR, 'GRS_AGEB_urbana_2020.xlsx')
for p in (imu_path, grs_path):
    assert os.path.exists(p), p

DEP = mx.load_deprivation(imu_path, grs_path)

T = T.merge(DEP[['CVEGEO', 'grs_grade', 'grs_grade_ord', 'imu_depriv',
                 'imu_depriv_z', 'imu_grade', 'has_imu', 'has_grs']],
            on='CVEGEO', how='left')

print()
print(f'tracts with a social backwardness grade  '
      f'{int(T.grs_grade.notna().sum()):,} ({T.grs_grade.notna().mean():.1%})')
print(f'tracts with a marginalisation score      '
      f'{int(T.imu_depriv.notna().sum()):,} ({T.imu_depriv.notna().mean():.1%})')
print(f'tracts with neither                      '
      f'{int((T.grs_grade.isna() & T.imu_depriv.isna()).sum()):,}')

print('\npopulation and vehicle availability by grade:\n')
print(T.groupby('grs_grade', dropna=False)
       .agg(tracts=('CVEGEO', 'size'), population=('POBTOT', 'sum'),
            median_share_with_car=('veh_share', 'median'))
       .reindex(mx.GRADE_ORDER + [np.nan]).round(4).to_string())

## 12. Attrition

Two separate accounts, because blocks and tracts are different units and a
single sequence that changes unit midway reports a difference between
quantities that are not comparable. Each step within an account is a subset of
the step above it.

In [ ]:
mx.flow('blocks in the geographic frame', len(BLOCKS), 'from the boundary clip')
mx.flow('blocks in the urban census', len(B), 'absent blocks are not urban')
mx.flow('blocks with inhabitants', int((B.POBTOT > 0).sum()),
        'uninhabited blocks retained and flagged')

print()
weightable = T['block_weightable']
graded = T['grs_grade'].notna()
both = graded & T['imu_depriv'].notna()

steps = [
    ('tracts with a block in the urban census', pd.Series(True, index=T.index)),
    ('tracts with at least one inhabited block', weightable),
    ('of those, with a deprivation grade', weightable & graded),
    ('of those, with both deprivation measures', weightable & both),
]
rows = []
prev = None
for label, mask in steps:
    n = int(mask.sum())
    rows.append({'stage': label, 'tracts': n,
                 'change': '' if prev is None else f'{n - prev:+,}',
                 'population': int(T.loc[mask, 'POBTOT'].sum())})
    prev = n
TRACT_FLOW = pd.DataFrame(rows)
print(TRACT_FLOW.to_string(index=False))

rep.add_table('block attrition', mx.flow_table(), max_rows=20)
rep.add_table('tract attrition', TRACT_FLOW, max_rows=20)

## 13. Checks

Each check names the condition under which it fails.

In [ ]:
print('--- checks ---\n')

# A block appearing twice would count its population twice.
d = int(B['CVEGEO_MZA'].duplicated().sum())
print(f'1. duplicate blocks after the join           {d}')
assert d == 0

# A tract in two areas would make tract level reporting ambiguous.
m = int((B.groupby('CVEGEO')['slug'].nunique() > 1).sum())
print(f'2. tracts in more than one area              {m}')
assert m == 0

# A difference here would mean the aggregation lost or repeated rows.
gap = abs(T.POBTOT.sum() - B.POBTOT.sum())
print(f'3. population, tracts against blocks         {gap:,.0f}')
assert gap < 1

# An area with no blocks in the census would be absent from all later results.
lost = sorted(set(BLOCKS.slug) - set(B.slug))
print(f'4. areas with no block in the census         {len(lost)}')
assert not lost, lost

# A share of dwellings cannot exceed one.
bad = int((T['veh_share'] > 1).sum())
print(f'5. tracts with a vehicle share above one     {bad}')
assert bad == 0

# Parks, industrial blocks and cemeteries carry no inhabitants, so a share of
# uninhabited blocks near zero would indicate the join had dropped them.
print(f'6. blocks with no inhabitants                {(B.POBTOT == 0).mean():.1%}')

# Metropolitan boundaries cross tracts, so a count of zero here would mean the
# completeness measure is not being computed.
cut = int((T['tract_complete'].fillna(1.0) < 0.95).sum())
print(f'7. tracts below 95 per cent complete         {cut:,} '
      f'({cut / len(T):.2%})')

print(f'\n8. population in the frame                   {T.POBTOT.sum():,.0f}')
print(f'9. median tract population                   {T.POBTOT.median():,.0f}')
print(f'10. median share of dwellings with a car     {T.veh_share.median():.4f}')
print(f'11. tracts without an inhabited block        {n_bad}')

## 14. Results by metropolitan area

In [ ]:
by_metro = (T.groupby('slug')
              .agg(tracts=('CVEGEO', 'size'), blocks=('blocks', 'sum'),
                   population=('POBTOT', 'sum'),
                   with_grade=('grs_grade', lambda s: int(s.notna().sum())),
                   median_density=('pop_density_km2', 'median'),
                   median_share_with_car=('veh_share', 'median'))
              .reset_index())

metros = mx.load_metros()
by_metro = by_metro.merge(
    metros.assign(slug=mx._slug_name(metros['metro_key']).values)
          [['slug', 'metro_name', 'tipo', 'tier', 'conapo_pop_2020']], on='slug')

# The published metropolitan population covers whole municipalities, including
# their rural parts, while this frame holds urban blocks, so the ratio is the
# urban share of each area's population.
by_metro['urban_pop_share'] = (
    by_metro['population'] / by_metro['conapo_pop_2020']).round(3)

print('--- per area, largest population first ---\n')
print(by_metro.sort_values('population', ascending=False)
             [['slug', 'metro_name', 'tipo', 'tracts', 'blocks', 'population',
               'conapo_pop_2020', 'urban_pop_share', 'median_density',
               'median_share_with_car']].to_string(index=False))

print('\n--- by official classification ---\n')
print(by_metro.groupby('tipo')
        .agg(areas=('slug', 'size'), tracts=('tracts', 'sum'),
             population=('population', 'sum'))
        .reindex(mx.TIPO_ORDER).to_string())

rep.add_table('per area totals', by_metro, max_rows=92)

## 15. Write results

In [ ]:
keep_b = ['CVEGEO_MZA', 'CVEGEO', 'CVE_ENT', 'MZA', 'slug', 'metro_key',
          'macro_region', 'area_m2'] + NUMERIC
outb = os.path.join(mx.DIRS['results'], '04_blocks.parquet')
B[keep_b].to_parquet(outb, index=False)
print('written:', outb, f'({os.path.getsize(outb) / 1e6:.1f} MB)')

outt = os.path.join(mx.DIRS['results'], '04_tracts.parquet')
T.to_parquet(outt, index=False)
print('written:', outt, f'({os.path.getsize(outt) / 1e6:.1f} MB)')

mx.save_flow(os.path.join(mx.DIRS['logs'], '04_sample_flow.csv'))

rep.note(f'{len(B):,} blocks and {len(T):,} tracts; {T.POBTOT.sum():,.0f} '
         f'people; vehicle denominator {DENOM}; median share without a car '
         f'{1 - T.veh_share.median():.3f}; '
         f'{int(T.grs_grade.notna().sum()):,} tracts with a deprivation grade')
rep.save()